# 03 — Measured Greater Cairo traffic

This notebook characterizes the official 2024 Transport for Cairo (TfC) directional link-count release. It reads only a locally prepared, hash-verified bundle; it never downloads data or substitutes a synthetic fixture. The project author reports direct TfC permission for academic research and data reuse. That permission is not an open license, and source or derived public redistribution remains bounded by the exact private terms; source and row-level files stay outside Git.

Evidence boundary: the records are **20-minute directional link counts**, not turning movements, signal timings, queues, travel times, or proof of a calibrated Cairo traffic model. Direction codes `0` and `1`, survey timezone, and sample-validity flags are not defined by the publisher.


## Prepare the local bundle on Windows PowerShell 5.1

Download the two official archives manually from the URLs documented in `docs/real_data_feasibility.md`, verify that the exact private permission covers the intended research use, then run from the repository root with Python 3.12:

```powershell
Set-Location "D:\UrbanTwin-AI\UrbanTwin-AI"
.\.venv\Scripts\Activate.ps1
$Counts = "C:\path\to\2024_traffic_counts.zip"
$Locations = "C:\path\to\2024_traffic_survey_locations.zip"
urbantwin observed-data-prepare `
  --counts $Counts `
  --locations $Locations `
  --output-dir "data\observed\tfc\normalized_v1" `
  --source-retrieved-at-utc "2026-11-08T10:54:38Z"
```

Use the real retrieval time in UTC. The command refuses to overwrite an existing inspected bundle.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from urbantwin.common import load_yaml
from urbantwin.data.observed_study import load_normalized_counts
from urbantwin.observed.features import build_analysis_table, historic_eo_context

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter from the UrbanTwin-AI repository root.")

STUDY_DIR = ROOT / "data/observed/tfc/normalized_v1"
MANIFEST_PATH = STUDY_DIR / "source_manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The verified TfC bundle is missing. Run the PowerShell preparation command above."
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest.get("data_kind") != "real_observed_traffic":
    raise ValueError("The local bundle is not labeled real observed traffic.")
if manifest.get("license", {}).get("redistribution_permitted") is not False:
    raise ValueError("Expected public redistribution to remain false until the exact private permission terms authorize it.")

counts = load_normalized_counts(STUDY_DIR)
analysis = build_analysis_table(STUDY_DIR)
config = load_yaml(ROOT / "configs/observed_traffic.yaml")
permission = config.get("permission", {})
if permission.get("status") != "author_reported_academic_research_reuse_authorized":
    raise ValueError("The configuration does not record the author-reported TfC research permission.")
if permission.get("public_redistribution_status") != "requires_exact_written_terms":
    raise ValueError("The configuration must not imply an unverified public-redistribution grant.")


## Provenance and release coverage

The loader rechecks artifact hashes, row counts, integer/nonnegative counts, totals, timezone status, sample-validity status, and duplicate observation IDs before returning data.


In [ ]:
coverage = pd.Series(
    {
        "publisher": manifest["provider"],
        "portal license status": manifest["license"]["status"],
        "portal/source-manifest public redistribution flag": manifest["license"]["redistribution_permitted"],
        "author-reported permission": permission["status"],
        "public redistribution status": permission["public_redistribution_status"],
        "directional intervals": len(counts),
        "directional segments": counts["segment_id"].nunique(),
        "inferred analytical sites": analysis["inferred_site_id"].nunique(),
        "first survey date": counts["observed_date"].min(),
        "last survey date": counts["observed_date"].max(),
        "observed vehicles": int(counts["observed_vehicle_total"].sum()),
        "ambiguous OSM-join rows": int(
            (~analysis["eligible_for_spatial_benchmark"]).sum()
        ),
    },
    name="value",
)
coverage.to_frame()


The 60-site count is reproduced by a declared 250 m connected-component rule in EPSG:32636. It is an analytical grouping, **not an official TfC `location_id`**. Five conflicting count/geometry OSM IDs remain visible and are excluded from the primary spatial benchmark rather than silently rewritten.


## Observed vehicle-class composition

These are source totals, not modeled estimates. No missing observation is imputed.


In [ ]:
vehicle_classes = manifest["normalization"]["vehicle_classes"]
class_totals = counts[vehicle_classes].sum().rename("observed_vehicles").to_frame()
class_totals["share"] = (
    class_totals["observed_vehicles"] / class_totals["observed_vehicles"].sum()
)
class_totals.sort_values("observed_vehicles", ascending=False)


## Predeclared Tahrir-AOI link-flow site

The selected 6th of October Bridge pair was chosen because it intersects the pre-existing AOI and has complete same-day coverage in both published directions. It is a motorway link-flow/EO-context site, not a signalized-intersection study site.


In [ ]:
selected_ids = config["site_selection"]["selected_segment_ids"]
selected = analysis.loc[analysis["segment_id"].isin(selected_ids)].copy()
selected_counts = selected.groupby("direction_code")["observation_id"].size()
if set(selected_counts) != {"0", "1"} or not selected_counts.eq(72).all():
    raise ValueError("Selected link coverage no longer matches the frozen study rule.")

figure, axis = plt.subplots(figsize=(10, 4.5))
for direction, rows in selected.groupby("direction_code", sort=True):
    rows = rows.sort_values("minute_of_day")
    axis.plot(
        rows["minute_of_day"] / 60,
        rows["observed_vehicle_total"],
        label=f"Direction code {direction}",
    )
axis.set(
    title="Measured 20-minute directional flow — 31 October 2024",
    xlabel="Publisher clock time (timezone not specified)",
    ylabel="Observed vehicles per 20 minutes",
)
axis.legend()
axis.grid(alpha=0.2)
figure.tight_layout()
plt.show()


## Retrospective Earth-observation context

The observed EO bundle contains Sentinel-2 L2A scenes from 24 and 29 October and 3 November 2024, with ESA WorldCover 2021 retained as a historic reference. The 3 November acquisition postdates the selected 31 October traffic survey, so these buffer statistics are retrospective spatial context, not prospective traffic-prediction inputs. EO is absent from the multi-site benchmark; no EO accuracy gain was tested.


In [ ]:
EO_DIR = ROOT / "data/observed/eo_2024"
if not (EO_DIR / "eo_stack.npz").is_file():
    raise FileNotFoundError(
        "The provenance-verified observed 2024 EO bundle is missing. Run observed-eo-fetch; do not substitute fake data."
    )
eo_context = pd.DataFrame(
    historic_eo_context(STUDY_DIR, EO_DIR, selected_ids, buffer_distances_m=(50, 100, 250))
)
eo_context[
    [
        "buffer_m",
        "valid_pixel_coverage",
        "mean_ndvi",
        "mean_ndbi",
        "worldcover_built_up_fraction",
        "worldcover_vegetated_fraction",
        "worldcover_year",
    ]
]


## Interpretation limits

- Every exact directional segment was surveyed on one date; repeated same-site temporal validation is unavailable.
- The public release has no official physical-site key, timezone, direction-code legend, or sample-validity field.
- The released survey lines are OSM-derived geometry from a 2026 extract, linked to 2024 counts; they are not a contemporaneous controller inventory.
- The observed 2024 EO bundle covers the selected Tahrir-area links only; the 3 November scene is post-survey retrospective context, and EO cannot support a 60-site model ablation or an accuracy-gain claim.
- No intersection turning movements or observed signal plans exist, so this notebook does not construct a calibrated SUMO demand matrix or a traffic controller.
